In [ ]:
# # 九曜：天墟 LF01 — 國漫配音 第 6 版：蕭曜霖 A10b 重做（Qwen3-TTS VoiceDesign，Kaggle）2026-10-01
#
# 咖哩試聽 final：「半刻鐘後，新生演武場集合。」變粵語。該句來自第 2 版（繁體提示詞）。
# 本版用同一提示詞的簡體版、6 個種子重唸；另出 A10a 簡體 s11 對照，確認簡體後聲音是否仍與選定的蕭曜霖一致。


In [ ]:
import subprocess
print(subprocess.run("pip install -q -U qwen-tts soundfile 2>&1 | tail -3; nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)


In [ ]:
import json, os, base64
ROWS = json.loads(r'''[["LF01-A10b_蕭曜霖_簡", "热血玄幻国漫（3D修仙动画）里的强者教官。四十多岁男声，低沉、沙哑、胸腔共鸣厚，带颗粒感。标准普通话，专业配音演员。声音不大却压迫感十足，冷峻、不容置疑，一字一顿，每个字都像落在地上，说完不留余地。", "半刻钟后，新生演武场集合。", [11, 22, 33, 44, 55, 66]], ["LF01-A10a_蕭曜霖_簡", "热血玄幻国漫（3D修仙动画）里的强者教官。四十多岁男声，低沉、沙哑、胸腔共鸣厚，带颗粒感。标准普通话，专业配音演员。声音不大却压迫感十足，冷峻、不容置疑，一字一顿，每个字都像落在地上，说完不留余地。", "不用等明天。", [11]]]''')
OUT = "/kaggle/working/voice_guoman"; os.makedirs(OUT, exist_ok=True)


In [ ]:
# VoiceDesign 直接唸（ROWS：名稱、提示詞、台詞、種子）
import torch, soundfile as sf, gc
from qwen_tts import Qwen3TTSModel
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
for name, prompt, text, seeds in ROWS:
    for s in seeds:
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        wavs, sr = design.generate_voice_design(text=text, language="Chinese", instruct=prompt)
        sf.write(f"{OUT}/{name}_s{s}.wav", wavs[0][: int(sr * 15)], sr)
        print(name, f"s{s}", f"{len(wavs[0])/sr:.2f}s", text)


In [ ]:
# （本版不用 Base 照唸）


In [ ]:
# 乾聲＋混音版 MP3，並以 base64 印進執行紀錄（Claude 從雲端抓不到 Output 檔）
import glob, base64, shutil
MIX = "equalizer=f=110:t=q:w=1:g=5,equalizer=f=3500:t=q:w=1.5:g=2,acompressor=threshold=-20dB:ratio=4:attack=5:release=120:makeup=4,aecho=0.8:0.6:60|120:0.25|0.15,loudnorm=I=-16"
dst = "/kaggle/working/voice_guoman_tiny"; os.makedirs(dst, exist_ok=True)
for w in sorted(glob.glob(f"{OUT}/*.wav")):
    b = f"{dst}/{os.path.basename(w)[:-4]}"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-ac", "1", "-b:a", "64k", b + ".mp3"], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-af", MIX, "-ac", "1", "-b:a", "64k", b + "_混音.mp3"], check=True)
shutil.make_archive("/kaggle/working/LF01_voice_guoman_tiny", "zip", dst)
for m in sorted(glob.glob(f"{dst}/*.mp3")):
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
